# Exploring Transformers: Tokenization, Self-Attention, and Text Generation with ModernBERT, SmolLM2 and a Frontier GPT Model

## Introduction
Transformers are the architecture behind every modern language model. This notebook explores four ideas that explain how they work: **tokenization**, **positional encoding**, **self-attention** and **text generation**.


### Why three models?
- **Small open models** (ModernBERT and SmolLM2) run inside this notebook. Because the weights are on your machine, you can look at their tokens, their attention patterns and their next-word probabilities.
- **Frontier models** are far larger and are only reachable through an API. You can see what they write, but not how. We use one at the end as a quality reference.

---

## 1. Tokenization
Tokenization splits text into smaller units called tokens. Tokens can be whole words, parts of words or single characters. Each token is mapped to a number, and those numbers are what the model actually reads.

- **BERT** used a **WordPiece** tokenizer with a vocabulary of about 30,000 lowercase pieces.
- **ModernBERT** uses a **Byte Pair Encoding (BPE)** tokenizer with a vocabulary of about 50,000 pieces and keeps upper and lower case.
- **Recent GPT models** use BPE with a vocabulary of about 200,000 pieces.

---

## 2. Self-Attention
Self-attention lets every token look at the other tokens in the sentence and decide which ones matter for its own meaning.

- **Encoders** such as ModernBERT use **bidirectional** attention: every token can see the words on both sides.
- **Decoders** such as SmolLM2 and the GPT family use **causal** attention: every token can only see the words before it. This is what makes it possible to generate text one token at a time.

---

## 3. Text Generation
- **Encoders** are built for understanding text (search, classification, retrieval). They are not designed to write.
- **Decoders** generate text by repeatedly predicting the next token and adding it to the sequence.
- A frontier model does exactly the same thing as the small decoder in this notebook. It is simply much larger and trained on far more data.


### 1. Installing Packages

This cell installs only the packages the notebook actually uses:

- **torch**: PyTorch, the deep learning framework that runs the models. It is already present on most SageMaker Studio images, in which case this is skipped.
- **transformers** (version 4.48 or later): the Hugging Face library used to load models and tokenizers. ModernBERT needs at least this version.
- **bertviz**: draws interactive attention diagrams.
- **tiktoken**: the open source tokenizer library published by OpenAI. It lets us use a real GPT tokenizer without calling any API.
- **matplotlib** and **ipywidgets**: plotting and notebook widgets.
- **openai**, **aws-bedrock-token-generator** and **botocore[crt]**: used only in the last section to call a frontier model on Amazon Bedrock.

**Note:** if this cell upgrades transformers, restart the kernel (Kernel menu, Restart Kernel) and continue from the next cell.


In [ ]:
%pip install --quiet torch 'transformers>=4.48' bertviz tiktoken matplotlib ipywidgets
%pip install --quiet --upgrade openai aws-bedrock-token-generator 'botocore[crt]'

print('All packages installed successfully!')

# Tokenizers

### 2. Loading the ModernBERT Model and Tokenizer

- **AutoTokenizer** and **AutoModel** are general loader classes from the transformers library. You give them a model name and they pick the right tokenizer and architecture.
- `answerdotai/ModernBERT-base` is the base size of ModernBERT, a modernised BERT-style encoder released in December 2024. It has 22 layers and about 149 million parameters, and it can read up to 8,192 tokens at once (BERT was limited to 512).
- `attn_implementation='eager'` tells the model to use the plain attention calculation. It is slightly slower, but it is the version that can return attention weights, which we need for the visualisation later.
- `model.eval()` switches the model to inference mode.
- `set_seed(42)` fixes the random seed so that sampled text is repeatable.
- The last lines print the number of layers, the number of attention heads per layer and the vocabulary size, so you can check them yourself.

The first run downloads the model, which takes a short while.


In [ ]:
import torch
import transformers
from transformers import AutoModel, AutoTokenizer, set_seed
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()
set_seed(42)

encoder_name = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(encoder_name)
model = AutoModel.from_pretrained(encoder_name, attn_implementation='eager')
model.eval()

print('transformers version:', transformers.__version__)
print('Loaded:', encoder_name)
print('Layers:', model.config.num_hidden_layers)
print('Attention heads per layer:', model.config.num_attention_heads)
print('Tokenizer vocabulary size:', tokenizer.vocab_size)

### 3. Tokenizing Input Text

- The sentence `I read a good novel.` is passed to the tokenizer.
- The result is a dictionary. The important entry is `input_ids`: one number for each token. These numbers are positions in the tokenizer vocabulary, and they are the only thing the model receives.
- `attention_mask` contains a 1 for every real token. It becomes useful when several sentences of different lengths are padded to the same length.
- You will see more IDs than there are words. The tokenizer adds a special token at the start and at the end of the sentence, and the full stop is a token of its own.


In [ ]:
sentence = 'I read a good novel.'
tokenized = tokenizer(sentence)
print(tokenized)

### 4. Understanding Tokenization and Tokens

- `tokenized['input_ids']` holds the token IDs from the previous cell.
- `convert_ids_to_tokens()` turns each ID back into the text piece it stands for, so you can see how the sentence was split.
- `[CLS]` and `[SEP]` are the special start and end tokens. ModernBERT keeps these from the original BERT.
- Pieces that begin a new word carry a marker for the space in front of them. This is how a BPE tokenizer remembers where the spaces were, so the original text can be rebuilt exactly.
- Notice that the capital `I` is preserved. The original `bert-base-uncased` model would have turned everything into lower case.


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
print(tokens)

### 5. Comparing Three Generations of Tokenizers

Simple sentences do not show much, because every word is already in the vocabulary. This cell uses a sentence with rarer words and runs it through three tokenizers:

- **BERT (2018), WordPiece**: loaded with `AutoTokenizer.from_pretrained('bert-base-uncased')`. Only the small tokenizer files are downloaded, not the model. Pieces that continue a word are marked with `##`.
- **ModernBERT (2024), BPE**: the tokenizer we loaded above.
- **OpenAI o200k_base, BPE**: loaded with `tiktoken.get_encoding('o200k_base')`. This is the tokenizer OpenAI published for GPT-4o through the GPT-5 series. It is open source, so no API key or payment is needed.

The helper function `show_tokens` prints the number of tokens and the pieces for each tokenizer. The last lines print the vocabulary sizes.

**What to look for:**
- A rare word is broken into several pieces. No tokenizer needs an entry for every word in the language.
- A larger vocabulary usually needs fewer tokens for the same text. Fewer tokens means less work for the model and, for paid APIs, a lower bill, because API prices are charged per token.


In [ ]:
import tiktoken

text = 'Tokenization of uncommon words like electroencephalogram is unbelievably interesting.'

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
gpt_encoding = tiktoken.get_encoding('o200k_base')

def show_tokens(name, pieces):
    print(name, '->', len(pieces), 'tokens')
    print(pieces)
    print()

show_tokens('BERT 2018 (WordPiece)', bert_tokenizer.tokenize(text))
show_tokens('ModernBERT 2024 (BPE)', tokenizer.tokenize(text))
show_tokens('OpenAI o200k_base (BPE)', [gpt_encoding.decode([i]) for i in gpt_encoding.encode(text)])

print('Vocabulary sizes')
print('BERT:', bert_tokenizer.vocab_size)
print('ModernBERT:', tokenizer.vocab_size)
print('OpenAI o200k_base:', gpt_encoding.n_vocab)

# Positional Encoding

### Explanation of the Code

A Transformer reads all tokens at the same time, so on its own it has no idea which token came first. Positional encoding adds that missing information.

This cell builds the **sinusoidal positional encoding** from the original Transformer paper (Attention Is All You Need, 2017). It is the clearest way to see the idea.

#### 1. Importing Required Libraries
- **numpy** handles the matrix of numbers.
- **matplotlib.pyplot** is used in the next cell to draw the matrix.

#### 2. The `encodePositions` Function
- **`num_tokens`**: how many positions to encode.
- **`depth`**: how many numbers describe each position.
- **`n=10000`**: a constant that sets the range of wavelengths.
- `positionalMatrix = np.zeros((num_tokens, depth))` creates an empty matrix with one row per position.

#### 3. Filling the Matrix
- The outer loop walks through the positions (rows).
- The inner loop walks through half of the columns, because each step fills two columns: one with a sine value and one with a cosine value.
- `denominator = np.power(n, 2*col/depth)` makes the wave slower as the column number grows. The first columns change quickly from one position to the next, and the last columns change very slowly.
- Even columns get `sin(row/denominator)` and odd columns get `cos(row/denominator)`.

#### 4. Result
Every row is a unique pattern, a kind of fingerprint for that position.

#### 5. What current models use
The models in this notebook do not use this exact method. BERT and GPT-2 learned their position vectors during training. ModernBERT, SmolLM2 and most recent models use **rotary position embeddings (RoPE)**, which rotate the vectors inside the attention calculation by an angle that depends on position. RoPE uses the same family of sine and cosine waves you see here, so the picture below is still the right mental model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def encodePositions(num_tokens, depth, n=10000):
    positionalMatrix = np.zeros((num_tokens, depth))
    for row in range(num_tokens):
        for col in np.arange(int(depth/2)):
            denominator = np.power(n, 2*col/depth)
            positionalMatrix[row, 2*col] = np.sin(row/denominator)
            positionalMatrix[row, 2*col+1] = np.cos(row/denominator)
    return positionalMatrix

### Explanation of the Code

1. **Generating the matrix**: `encodePositions(50, 256)` creates a matrix with 50 rows (positions) and 256 columns (encoding dimensions).
2. **Drawing it**: `plt.matshow(positionalMatrix)` shows the matrix as a heatmap. Each colour is one value between -1 and 1.
3. **Colour bar and labels**: `colorbar` adds the scale, and the axis labels show which direction is position and which is dimension.

**How to read the picture:** on the left the stripes change quickly from row to row, and towards the right they change more and more slowly. Reading one row from left to right gives a pattern that no other row has, which is how the model can tell positions apart.


In [ ]:
positionalMatrix = encodePositions(50, 256)
image = plt.matshow(positionalMatrix)
plt.gcf().colorbar(image)
plt.xlabel('Encoding dimension')
plt.ylabel('Token position')
plt.show()

# Self-Attention

### Explanation of the Code

1. **Importing the visualiser**
   - `from bertviz import head_view` imports the head view from the BertViz library. It works with any model that can return its attention weights, which is why we can use it with ModernBERT.

2. **The `encoder_attention` function**
   - `tokenizer(sentence, return_tensors='pt')` tokenizes the sentence and returns PyTorch tensors.
   - `torch.no_grad()` tells PyTorch we are only running the model, not training it.
   - `model(**inputs, output_attentions=True)` runs ModernBERT and asks it to return the attention weights.
   - `outputs.attentions` contains one tensor per layer. Each tensor has the shape (1, heads, tokens, tokens): for every head, how strongly each token looks at every other token.
   - `pieces` is the readable text of each token, used as labels in the diagram.

3. **Showing the attention**
   - `head_view(attention, pieces)` draws the sentence twice. A line from a token on the left to a token on the right means the left token is paying attention to the right one. A stronger line means more attention.

**Try it:**
- Use the **Layer** drop-down to move through the 22 layers. Early layers often show simple patterns, such as looking at the next or previous token. Later layers show patterns related to meaning.
- Click the coloured squares to turn individual **heads** on and off. Each head has learned to look for something different.
- Hover over the word `novel` and see which words it attends to. Here it is a noun, the thing being read.
- Because ModernBERT is an encoder, tokens attend to words on **both sides**.


In [ ]:
from bertviz import head_view

def encoder_attention(sentence):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    pieces = [tokenizer.decode([i]).strip() for i in inputs['input_ids'][0].tolist()]
    return outputs.attentions, pieces

attention, pieces = encoder_attention('I read a good novel.')
print('Layers returned:', len(attention))
print('Shape of one layer (batch, heads, tokens, tokens):', tuple(attention[0].shape))
head_view(attention, pieces)

### Explanation of the Code

This cell repeats the visualisation with a different sentence: `Attention is a novel idea.`

- The word `novel` appears again, but now it is an adjective meaning new, not a book.
- The token ID for `novel` is the same in both sentences. The model can only tell the two meanings apart by looking at the surrounding words, and attention is the mechanism that does this.

**Try it:** choose a later layer, hover over `novel`, and compare what it attends to here with what it attended to in the previous sentence. In the first sentence the useful context was `read`; here it is `idea`.


In [ ]:
attention, pieces = encoder_attention('Attention is a novel idea.')
head_view(attention, pieces)

# SmolLM2 (about 360M parameters)

### Explanation of the Code

GPT-2 was the last GPT model whose weights could run in a small notebook, and it dates from 2019. Here we use **SmolLM2**, a small open model from Hugging Face released in 2024. It follows the same decoder-only design as the GPT family, with the improvements used in current models.

1. **Freeing memory**
   - `del model` and `gc.collect()` remove ModernBERT from memory, because the lab instance is small. After this cell, the two attention cells above cannot be re-run unless you reload ModernBERT.

2. **Loading the decoder**
   - `AutoModelForCausalLM` loads a model for causal language modelling, which means predicting the next token.
   - `HuggingFaceTB/SmolLM2-360M` is the base version, which simply continues whatever text it is given, exactly like GPT-2 did.
   - `attn_implementation='eager'` is used again so that we can look at the attention weights.

3. **Creating the pipeline**
   - `pipeline('text-generation', ...)` wraps the model and tokenizer in a simple function: text in, text out. Passing the objects we already loaded avoids loading the model a second time.

4. **Counting parameters**
   - The last lines count the parameters so you can see the real size of the model.

If the kernel stops because it runs out of memory, restart it and change the model name to the smaller `HuggingFaceTB/SmolLM2-135M`.


In [ ]:
import gc
from transformers import AutoModelForCausalLM, pipeline

# Free the memory used by ModernBERT
del model
gc.collect()

decoder_name = 'HuggingFaceTB/SmolLM2-360M'
gen_tokenizer = AutoTokenizer.from_pretrained(decoder_name)
gen_model = AutoModelForCausalLM.from_pretrained(decoder_name, attn_implementation='eager')
gen_model.eval()

generator = pipeline('text-generation', model=gen_model, tokenizer=gen_tokenizer)

n_params = sum(p.numel() for p in gen_model.parameters())
print('Loaded:', decoder_name)
print('Parameters (millions):', round(n_params / 1e6))

### Explanation of the Code

Before generating whole sentences, this cell shows the single step that all text generation is built on: **predicting the next token**.

- The prompt `I read a good` is tokenized and passed through the model.
- `logits` contains one score for every token in the vocabulary, at every position.
- `logits[0, -1]` picks the scores at the last position, which are the scores for what comes next.
- `torch.softmax` turns the scores into probabilities that add up to 100 percent.
- `torch.topk(probs, 5)` selects the five most likely next tokens.

The model does not know the answer. It has a probability for every possible next token. Generation means choosing one of them, adding it to the text, and repeating. Every model in the GPT family, including the frontier model at the end of this notebook, works this way.

**Try it:** change the prompt and see how the candidates change.


In [ ]:
prompt = 'I read a good'
inputs = gen_tokenizer(prompt, return_tensors='pt')
with torch.no_grad():
    logits = gen_model(**inputs).logits

probs = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 5)

print('Prompt:', prompt)
print('Most likely next tokens:')
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(repr(gen_tokenizer.decode([i])), round(p * 100, 1), '%')

### Explanation of the Code

This cell draws the attention of the decoder for the same sentence we used with ModernBERT, so you can compare the two kinds of model.

- `gen_model(**inputs, output_attentions=True)` returns the attention weights of SmolLM2.
- `head_view` draws them in the same way as before.

**What to look for:** every line goes from a token to itself or to a token **earlier** in the sentence. No token attends to a later token. This is causal (one-directional) attention. The model is not allowed to see the future, because during generation the future has not been written yet.

Compare this with ModernBERT above, where tokens attended in both directions.


In [ ]:
inputs = gen_tokenizer('I read a good novel.', return_tensors='pt')
with torch.no_grad():
    outputs = gen_model(**inputs, output_attentions=True)

pieces = [gen_tokenizer.decode([i]).strip() for i in inputs['input_ids'][0].tolist()]
head_view(outputs.attentions, pieces)

### Explanation of the Code

Now the model generates full continuations of a prompt.

- **`'I read a good novel.'`** is the prompt. The model continues from here.
- **`max_new_tokens=30`** limits each continuation to 30 new tokens. The prompt is not counted.
- **`num_return_sequences=5`** asks for five different continuations of the same prompt.
- **`do_sample=True`** makes the model pick each next token at random according to its probabilities, instead of always taking the most likely one. This is why the five results differ.
- **`set_seed(42)`** makes the random choices repeatable, so running the cell again gives the same text.
- The helper function `show_outputs` prints the results in a readable way and is reused in the next cells.

A small model produces text that is grammatical but often drifts off topic. Keep these results in mind for the comparison with the frontier model at the end.


In [ ]:
def show_outputs(output):
    for idx, item in enumerate(output):
        print('Generated Text', idx + 1)
        print(item['generated_text'])
        print('=' * 50)

set_seed(42)
output = generator('I read a good novel.', max_new_tokens=30, num_return_sequences=5, do_sample=True)
show_outputs(output)

### Explanation of the Code

This cell generates longer text from a different prompt.

- **`'This movie seemed really long.'`** is the prompt.
- **`max_new_tokens=120`** allows up to 120 new tokens, about a paragraph.
- **`num_return_sequences=3`** asks for three continuations. Longer text takes more time on a small CPU instance, so we ask for fewer.

**What to look for:** the longer the text, the more likely a small model is to repeat itself or lose the thread. Staying coherent over long passages is one of the main things that larger models do better.

This cell can take a minute or two to run.


In [ ]:
set_seed(42)
output = generator('This movie seemed really long.', max_new_tokens=120, num_return_sequences=3, do_sample=True)
show_outputs(output)

### Explanation of the Code

This cell uses a very short prompt and adds two settings that control how the next token is chosen.

- **`'Star Trek'`** is the prompt. With so little context, the model has a lot of freedom.
- **`max_new_tokens=60`** allows up to 60 new tokens.
- **`temperature=0.7`**: a temperature below 1 makes likely tokens even more likely, so the text becomes more focused and predictable. A temperature above 1 does the opposite and makes the text more random.
- **`top_p=0.9`**: the model only chooses from the most likely tokens whose probabilities add up to 90 percent. The unlikely tail is cut off, which removes many strange word choices.

**Try it:** change the temperature to 1.5 and run the cell again, then try 0.2. The same model produces very different text depending on these two numbers.


In [ ]:
set_seed(42)
output = generator('Star Trek', max_new_tokens=60, num_return_sequences=5, do_sample=True, temperature=0.7, top_p=0.9)
show_outputs(output)

# Comparison with a frontier model

### Explanation of the Code

The model we used above simply continues text. Chat assistants are built on the same kind of model, with one extra step: the conversation is wrapped in a **chat template** before it is tokenized.

- `HuggingFaceTB/SmolLM2-360M-Instruct` is the chat-tuned version of SmolLM2. Here we load only its tokenizer, which is a small download.
- `messages` is a conversation in the standard format: a list of roles and contents.
- `apply_chat_template(..., tokenize=False)` shows the text that the model would actually receive.

**What to look for:** the output contains special marker tokens around the user message, and it ends with the start of the assistant turn. The model then continues the text from that point, one token at a time. A chat answer is still next-token prediction. This is the form in which frontier models receive your prompt.


In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')

messages = [{'role': 'user', 'content': 'Continue this text in two sentences: I read a good novel.'}]
print(chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

### Explanation of the Code

This cell sends the same three prompts to a frontier model and prints its answers next to the output of the small local model.

**The model.** `openai.gpt-6-luna` is GPT-6 Luna, the lowest-cost model of the OpenAI GPT-6 family on Amazon Bedrock. Its weights cannot be downloaded, so it runs on AWS servers and the notebook only sends a request.

**Before you run it.** The SageMaker execution role must be allowed to call Amazon Bedrock through the Bedrock Mantle endpoint, and the model is served from the US East (N. Virginia) Region. If the permission is missing, the cell prints a message instead of failing.

**Cost.** Bedrock charges per token. The three short requests in this cell cost well under one US cent in total.

**The code, step by step:**
- `provide_token(region=...)` creates a short-lived access token from the IAM role of this notebook. No API key is typed or stored in the notebook.
- `OpenAI(base_url=..., api_key=...)` creates a client with the OpenAI Python library and points it at the Bedrock endpoint instead of at OpenAI.
- `client.responses.create(model=..., input=...)` sends the request.
- `response.output_text` is the text of the answer.
- For each prompt, the small local model is also run once, so both results are printed together.

**Why the instruction is added.** A chat model answers a message instead of continuing it. To compare like with like, we ask it to continue the text in two sentences.

**What to compare:**
- Does each model stay on the topic of the prompt?
- Which text is more coherent from start to finish?
- Both models do the same thing, predict the next token. The difference in quality comes from scale: SmolLM2 has a few hundred million parameters, and frontier models are far larger and trained on far more text.

**What you cannot do here:** you cannot look at the tokenizer, the attention or the next-token probabilities of the frontier model. That is why the earlier sections used open models.


In [ ]:
%pip install aws-bedrock-token-generator 'botocore[crt]'

In [ ]:
from openai import OpenAI
from aws_bedrock_token_generator import provide_token

BEDROCK_REGION = 'us-east-1'
FRONTIER_MODEL = 'openai.gpt-oss-120b'

prompts = ['I read a good novel.', 'This movie seemed really long.', 'Star Trek']

try:
    client = OpenAI(
         base_url='https://bedrock-mantle.' + BEDROCK_REGION + '.api.aws/v1',
        api_key=provide_token(region=BEDROCK_REGION),
    )

    for prompt in prompts:
        set_seed(42)
        local_text = generator(prompt, max_new_tokens=40, do_sample=True)[0]['generated_text']

        response = client.responses.create(
            model=FRONTIER_MODEL,
            input='Continue this text in two sentences: ' + prompt,
        )

        print('PROMPT:', prompt)
        print()
        print('Small local model (' + decoder_name + '):')
        print(local_text)
        print()
        print('Frontier model (' + FRONTIER_MODEL + '):')
        print(response.output_text)
        print('=' * 70)

except Exception as error:
    print('The Bedrock call did not succeed.')
    print('Error:', error)
    print('Check that the SageMaker execution role is allowed to call Amazon Bedrock,')
    print('and that the model is available in the Region', BEDROCK_REGION)

### Conclusion

In this lab, we learned the following key concepts and techniques:

1. **Tokenization**
   - We tokenized text with ModernBERT and converted the token IDs back into readable pieces.
   - We compared three generations of tokenizers (BERT WordPiece, ModernBERT BPE and the OpenAI o200k_base encoding) and saw how vocabulary size affects the number of tokens.

2. **Positional Encoding**
   - We built and plotted the sinusoidal positional encoding from the original Transformer, and learned that current models use rotary position embeddings based on the same idea.

3. **Self-Attention**
   - We visualised attention in ModernBERT across layers and heads, and saw how the same word attends to different context in different sentences.
   - We compared the bidirectional attention of an encoder with the causal attention of a decoder.

4. **Text Generation**
   - We looked at the next-token probabilities of a GPT-style model and generated text with different lengths, temperature and top-p settings.
   - We saw how a chat template turns a conversation into plain text for the model.

5. **Comparison with a frontier model**
   - We sent the same prompts to a frontier model on Amazon Bedrock and compared its output with a small open model.

By the end of this lab, we gained hands-on experience with the building blocks of every modern language model, from the tokenizer to the generated text, using both models we can inspect and a model we can only call.
